# Patch Diffusion for QG Super-Resolution

## Imports

In [ ]:
import os
import sys
import math
import time
import timeit
import pickle
import importlib
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.ticker import MaxNLocator, ScalarFormatter, FixedLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from tqdm.auto import tqdm

## Paths and trained model

In [ ]:
sys.path.insert(0, '/home/cfyzhan/cfyzhan_data/cfyzhan/Diffusion_V0002/Code')
os.chdir('/home/cfyzhan/cfyzhan_data/cfyzhan/Diffusion_V0002/Code')

sys.path.append('/gdata/Results/cfyzhan/QG_V0003/Src')

from Grid.grid import Grid
from Operators.operators import SpectralDerivatives, SGSVars, NonlinearOperator
from Operators.spectral_conversion import to_physical, to_spectral, dealias, get_puv
from Plotting.plots import spectrum
from Initial_forcing.ics import int_sq

SAVE_DIR = "/gdata/Results/cfyzhan/Patch_Diff/Results"
os.makedirs(SAVE_DIR, exist_ok=True)

sys_dir = "/gdata/Results/cfyzhan/Diffusion_V0002/Results"
data_dir = "/gdata/Results/cfyzhan/QG_V0003/Results/"

run_number = 59
gen_number = 1
run_dir = os.path.join(sys_dir, f"Run{run_number:05d}")
gen_dir = os.path.join(run_dir, f"Gen{gen_number:05d}")
sys.path.append(os.path.join(run_dir, "Code"))

config_module = f'Config.Run{run_number:05d}'
config = importlib.import_module(config_module)

from Models.networks import UNet_large
model_cond = UNet_large(channels=[32, 64, 128, 256, 512], in_channels=2,
                         out_channels=1, dropout_rate=0,
                         attention=False, condition=True).cuda()

cond_model_load_directory = f'/gdata/Results/cfyzhan/Diffusion_V0002/Results/Run{run_number:05d}/Checkpoints/model_00040.pt'


def load_ddp_weights(model, ckpt_path, device="cpu", strict=True):
    state_dict = torch.load(ckpt_path, map_location=device)
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace("module.", "", 1) if k.startswith("module.") else k
        new_state_dict[name] = v
    missing, unexpected = model.load_state_dict(new_state_dict, strict=strict)
    print("loaded checkpoint:", ckpt_path)
    print("missing keys:", missing)
    print("unexpected keys:", unexpected)


load_ddp_weights(model_cond, cond_model_load_directory)
model_cond.eval()

## Diffusion helper functions

In [ ]:
def mu_sigma(t, option):
    if option == 0:
        omega = math.acos(math.sqrt(0.001))
        mu = torch.cos(omega * t) ** 2
    elif option == 1:
        mu = 1 - t
    mach_eps = 1e-2
    sigma = torch.sqrt(1 - mu ** 2 + mach_eps ** 2)
    return mu.float(), sigma.float()


def get_score(pred, x_noisy, t, option):
    _, sigma = mu_sigma(t, option)
    return -pred / sigma


def forward_sample(x, t, option):
    eps = torch.randn_like(x)
    mu, sigma = mu_sigma(t, option)
    mu = mu.view(-1, 1, 1).cuda()
    sigma = sigma.view(-1, 1, 1).cuda()
    x_out = mu * x + sigma * eps
    return x_out, eps

## Load QG data

In [ ]:
cond_run = 3480
time_index = 159
STD = 2.75

cond_data_dir = f"/gdata/Results/cfyzhan/QG_V0003/Results/Run{cond_run:05d}/fields_pooled512_ds64_Run{cond_run:05d}.npy"
cond_data_unpooled_dir = f"/gdata/Results/cfyzhan/QG_V0003/Results/Run{cond_run:05d}/fields_512_ds64_Run{cond_run:05d}.npy"
truth_data_dir = f"/gdata/Results/cfyzhan/QG_V0003/Results/Run{cond_run:05d}/fields_Run{cond_run:05d}.npy"

cond_data = np.load(cond_data_dir)
truth_data = np.load(truth_data_dir)
coarse_data = np.load(cond_data_unpooled_dir)

cond_data_T = (torch.from_numpy(cond_data[:, :, time_index, 0]).unsqueeze(0).float().cuda()) / STD
truth_data_T = (torch.from_numpy(truth_data[:, :, time_index, 0]).unsqueeze(0).float().cuda()) / STD
coarse_data_T = (torch.from_numpy(coarse_data[:, :, time_index, 0]).unsqueeze(0).float().cuda()) / STD

## Spin-up check

In [ ]:
n_time = truth_data.shape[2]
energy_series = np.array([np.mean(truth_data[:, :, t, 0] ** 2) for t in range(n_time)])
SPINUP = 25

fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.plot(np.arange(n_time), energy_series, "-", color="#1F4E79", lw=1.6, zorder=3)
ax.axvline(SPINUP, color="grey", ls="-.", lw=1.4, zorder=2)
ax.set_xlabel("Time index", fontsize=14)
ax.set_ylabel("Energy", fontsize=14)
ax.set_xlim(0, n_time - 1)
ax.set_ylim(0, float(energy_series.max()) * 1.08)
for sp in ax.spines.values():
    sp.set_edgecolor("black")
    sp.set_linewidth(1.6)
ax.grid(True, ls="--", alpha=0.25, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "fig_dynamic_balance.png"), dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

## Spectral reference and evaluation metrics

In [ ]:
LES_Nx = 64
LES_Ny = LES_Nx
DNS_Nx = 512
DNS_Ny = DNS_Nx

grid_LES = Grid(Nx=LES_Nx, Ny=LES_Ny)
spec_der_LES = SpectralDerivatives(grid_LES)
nlop_LES = NonlinearOperator(spec_der_LES, None)

grid_DNS = Grid(Nx=DNS_Nx, Ny=DNS_Ny)
spec_der_DNS = SpectralDerivatives(grid_DNS)
nlop_DNS = NonlinearOperator(spec_der_DNS, None)

scale = DNS_Nx // LES_Nx


def get_spectra(field, spectral_derivative):
    qh_sol = to_spectral(field)
    ph_sol, uh_sol, vh_sol = get_puv(qh_sol, spectral_derivative)
    z = torch.abs(qh_sol) ** 2
    e = torch.abs(uh_sol) ** 2 + torch.abs(vh_sol) ** 2
    k, [ek, zk] = spectrum([e, z], spectral_derivative)
    return k, [ek, zk]


def _to_np(a):
    if isinstance(a, torch.Tensor):
        a = a.detach().cpu().numpy()
    return np.asarray(a)


def _as_field(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().squeeze().cpu().numpy()
    else:
        x = np.asarray(x).squeeze()
    if x.ndim != 2:
        raise ValueError(f"expected a 2D field after squeeze, got {x.shape}")
    return x.astype(np.float64)


def reference_spectra(truth, spec_der, device="cuda"):
    field = truth if isinstance(truth, torch.Tensor) else torch.as_tensor(truth)
    k, [ek, zk] = get_spectra(field.squeeze().float().to(device), spec_der)
    return {"k": _to_np(k), "ek": _to_np(ek), "zk": _to_np(zk)}


def spectral_errors(pred, spec_der, ref, kmax=None, eps=1e-20, device="cuda"):
    field = pred if isinstance(pred, torch.Tensor) else torch.as_tensor(pred)
    _, [ek, zk] = get_spectra(field.squeeze().float().to(device), spec_der)
    out = {}
    for name, cur, refk in (("spec_log_norm", ek, ref["ek"]), ("spec_log_norm_z", zk, ref["zk"])):
        a = np.log(np.clip(_to_np(cur), eps, None))
        b = np.log(np.clip(refk, eps, None))
        if kmax is not None:
            a, b = a[:kmax], b[:kmax]
        out[name] = float(np.linalg.norm(b - a) / np.sqrt(len(b)))
    return out


def field_errors(pred, truth):
    p, t = _as_field(pred), _as_field(truth)
    if p.shape != t.shape:
        raise ValueError(f"shape mismatch: pred {p.shape} vs truth {t.shape}")
    d = p - t
    frob = float(np.linalg.norm(d))
    return {
        "notebook_norm": frob / np.sqrt(t.shape[0]),
        "rmse": float(np.sqrt(np.mean(d ** 2))),
        "mae": float(np.mean(np.abs(d))),
        "rel_l2": frob / float(np.linalg.norm(t)),
        "max_abs": float(np.abs(d).max()),
        "frobenius": frob,
    }


def bandpass(field, klo, khi):
    n = field.shape[0]
    k = np.fft.fftfreq(n, 1 / n)
    K = np.hypot(*np.meshgrid(k, k, indexing="xy"))
    return np.real(np.fft.ifft2(np.fft.fft2(field) * ((K >= klo) & (K <= khi))))


BANDS = [(4, 16), (16, 32), (32, 64), (64, 128)]


def band_stats(pred, truth, bands=BANDS):
    out = {}
    for lo, hi in bands:
        a, d = bandpass(pred, lo, hi), bandpass(truth, lo, hi)
        nd = np.linalg.norm(d)
        out[f"eps{lo}_{hi}"] = float(np.linalg.norm(a - d) / nd)
        out[f"cor{lo}_{hi}"] = float(np.corrcoef(d.ravel(), a.ravel())[0, 1])
        out[f"en{lo}_{hi}"] = float((np.linalg.norm(a) / nd) ** 2)
    return out


def _conv2d_same(field, kernel):
    t = torch.from_numpy(field).float().unsqueeze(0).unsqueeze(0)
    k = torch.from_numpy(kernel).float().unsqueeze(0).unsqueeze(0)
    return F.conv2d(t, k, padding=1).squeeze().numpy()


def seam_score(field, stride, band=2):
    f = _as_field(field)
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
    grad_mag = np.abs(_conv2d_same(f, sobel_x))

    H, W = f.shape
    seam_mask = np.zeros(W, dtype=bool)
    for c in range(0, W, stride):
        lo, hi = max(0, c - band), min(W, c + band + 1)
        seam_mask[lo:hi] = True

    seam_grad = grad_mag[:, seam_mask].mean()
    nonseam_grad = grad_mag[:, ~seam_mask].mean()
    return {"seam_grad": float(seam_grad), "nonseam_grad": float(nonseam_grad),
            "seam_ratio": float(seam_grad / nonseam_grad)}

## Patch SDEdit sampler

In [ ]:
PAPER_STRIDES = (8, 16, 32, 48, 64, 80, 96, 112, 128)
PAPER_SDEDIT = tuple(range(0, 65, 8))


def _chunked_forward(model, x, t, cond, chunk):
    n = x.shape[0]
    if chunk is None or n <= chunk:
        return model(x, t, cond)
    outs = [model(x[i:i + chunk], t, cond[i:i + chunk]) for i in range(0, n, chunk)]
    return torch.cat(outs, dim=0)


def run_sdedit(model, cond, stride, sdedit_time,
               seed=20, T=64, chunk=256, fast=True,
               option=0, num_corrections=1, corr_scale=0.25,
               patch_size=128, sigma_w=10000.0):
    torch.manual_seed(seed)

    device = cond.device
    H, W = cond.shape[-2], cond.shape[-1]
    ps = patch_size

    t_i = (sdedit_time / T) * torch.ones(1, device=device)
    x, _ = forward_sample(cond, t_i, option)
    loop_init = T - sdedit_time + 1

    with torch.no_grad():
        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, ps, device=device),
            torch.linspace(-1, 1, ps, device=device),
            indexing="ij",
        )
        patch_weight = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sigma_w ** 2))
        patch_weight = patch_weight / patch_weight.max()

        positions = list(range(0, H, stride))
        P = len(positions)
        N = P * P

        mu_i, sigma_i = mu_sigma(t_i, option)

        pos = torch.tensor(positions, device=device)
        ar = torch.arange(ps, device=device)
        row_idx = (pos[:, None] + ar[None, :]) % H
        col_idx = (pos[:, None] + ar[None, :]) % W

        def gather(field2d):
            out = field2d[row_idx[:, None, :, None], col_idx[None, :, None, :]]
            return out.reshape(N, ps, ps)

        def scatter(tiles, accum):
            step = chunk if chunk else N
            for a in range(0, N, step):
                b = min(a + step, N)
                idx = torch.arange(a, b, device=device)
                r = row_idx[idx // P]
                c = col_idx[idx % P]
                rr = r[:, :, None].expand(-1, ps, ps).reshape(-1)
                cc = c[:, None, :].expand(-1, ps, ps).reshape(-1)
                accum.index_put_((rr, cc), tiles[a:b].contiguous().reshape(-1), accumulate=True)

        patches_cond = gather(cond[0])
        weight = torch.zeros(H, W, device=device)
        scatter(patch_weight.expand(N, ps, ps), weight)
        weight = torch.clamp(weight, min=1e-8)

        for i in range(loop_init, T + 1):
            t_i_dash = torch.tensor(1 - i / T, device=device)
            mu_i_dash, sigma_i_dash = mu_sigma(t_i_dash, option)
            r = mu_i_dash / mu_i

            patches = gather(x[0])
            pred = _chunked_forward(model, patches, t_i, patches_cond, chunk)

            if i != T:
                x_i_1 = r * patches + (r - sigma_i_dash / sigma_i) * (sigma_i ** 2) \
                    * get_score(pred, patches, t_i, option)
            else:
                x_i_1 = patches + (sigma_i ** 2) * get_score(pred, patches, t_i, option)

            for _ in range(num_corrections):
                z = torch.randn_like(x_i_1)
                pred = _chunked_forward(model, x_i_1, t_i_dash, patches_cond, chunk)
                eps = get_score(pred, x_i_1, t_i_dash, option)
                denom = torch.clamp(F.mse_loss(eps, torch.zeros_like(eps)), min=1e-8)
                delta = corr_scale / denom
                x_i_1 = x_i_1 + delta * eps + torch.sqrt(2 * delta) * z

            x_next = torch.zeros(H, W, device=device)
            scatter(x_i_1 * patch_weight, x_next)
            x = (x_next / weight).unsqueeze(0)

            mu_i, sigma_i, t_i = mu_i_dash, sigma_i_dash, t_i_dash

        out = x.squeeze().detach().cpu().clone()

    del x
    torch.cuda.empty_cache()
    return out


def n_patches(stride, domain=512):
    return len(range(0, domain, stride)) ** 2


BASE = dict(T=64, chunk=256, fast=True, option=0, num_corrections=1, corr_scale=0.25)
BEST_STRIDE = 32
BEST_T0 = 16
SIGMA_W = 0.5

## Baseline generation at the selected condition

In [ ]:
cmap_arr = np.load('/gdata/Results/cfyzhan/QG_V0003/Src/Plotting/bor.npy')
bor = ListedColormap(cmap_arr, name="bor")

generated = run_sdedit(model_cond, cond_data_T, BEST_STRIDE, BEST_T0,
                        seed=20, sigma_w=SIGMA_W, **BASE)

fdns_err = field_errors(cond_data_T, truth_data_T)
gen_err = field_errors(generated, truth_data_T)
print(f"FDNS  error norm: {fdns_err['notebook_norm']:.4f}")
print(f"model error norm: {gen_err['notebook_norm']:.4f}")

## Field comparison plot

In [ ]:
def add_inset_cbar(im, ax):
    cax = inset_axes(ax, width="3.5%", height="100%",
                      bbox_to_anchor=(1.04, 0, 1, 1), bbox_transform=ax.transAxes,
                      loc="lower left", borderpad=0)
    return ax.figure.colorbar(im, cax=cax)


def get_plot(field, cmap_color, vmax, vmin, cmap_display, title=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))
    else:
        fig = ax.get_figure()
    ax.set_box_aspect(1)
    if title is not None:
        ax.set_title(title, fontsize=12, fontfamily='serif', fontstyle='italic')
    im = ax.imshow(field, cmap=cmap_color, origin='lower', vmax=vmax, vmin=vmin)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(2.0)
    if cmap_display:
        cbar = add_inset_cbar(im, ax)
        cbar.locator = MaxNLocator(nbins=3)
        sf = ScalarFormatter(useMathText=True)
        sf.set_scientific(True)
        sf.set_powerlimits((-1, 100))
        cbar.update_ticks()
        for spine in cbar.ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1.0)
        for label in cbar.ax.get_yticklabels():
            label.set_fontsize(10)
        cbar.ax.yaxis.set_major_formatter(sf)
    return im


fig = plt.figure(figsize=(12, 8))
plt.rcdefaults()

gs = fig.add_gridspec(nrows=2, ncols=4, width_ratios=[1, 1, 1, 0.1],
                       height_ratios=[1, 1], wspace=0.05, hspace=0.05)
vmax = 2.5

im = get_plot(truth_data_T.squeeze().cpu(), bor, vmax, -vmax, False, title='DNS', ax=fig.add_subplot(gs[0, 0]))
im = get_plot(cond_data_T.squeeze().cpu(), bor, vmax, -vmax, False, title='FDNS', ax=fig.add_subplot(gs[0, 1]))
im = get_plot(generated, bor, vmax, -vmax, False, title='Generative Model', ax=fig.add_subplot(gs[0, 2]))
im = get_plot((truth_data_T.squeeze().cpu() - cond_data_T.squeeze().cpu()), bor, vmax, -vmax, False,
              title='FDNS Error', ax=fig.add_subplot(gs[1, 1]))
im = get_plot((truth_data_T.squeeze().cpu() - generated), bor, vmax, -vmax, False,
              title='Generative Model Error', ax=fig.add_subplot(gs[1, 2]))

cax = fig.add_subplot(gs[:, -1])
cbar = fig.colorbar(im, cax=cax)
for spine in cbar.ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.0)
for label in cbar.ax.get_yticklabels():
    label.set_fontsize(12)
cbar.locator = MaxNLocator(nbins=5)
sf = ScalarFormatter(useMathText=True)
sf.set_scientific(True)
sf.set_powerlimits((-1, 1))
cbar.ax.yaxis.set_major_formatter(sf)
cbar.update_ticks()

fig.savefig(os.path.join(SAVE_DIR, "fig_sr_fields.png"), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## Energy spectrum comparison

In [ ]:
k_true, [ek_true, zk_true] = get_spectra(truth_data_T.squeeze(), spec_der_DNS)
k_pred, [ek_pred, zk_pred] = get_spectra(generated.cuda(), spec_der_DNS)
k_cond, [ek_cond, zk_cond] = get_spectra(cond_data_T.squeeze(), spec_der_DNS)

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(k_true, ek_true, color='black', label='DNS')
ax.loglog(k_cond[:31], ek_cond[:31], color='grey', label='FDNS', ls='-.')
ax.loglog(k_pred, ek_pred, color='red', label='Generative Model', ls='--')
ax.set_ylim(1e-13, 1e0)
ax.set_xlabel("k", fontsize=16)
ax.set_ylabel("E(k)", fontsize=16)
ax.legend()
for spine in ax.spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(2)
ax.grid(True, which='major', axis='y', ls='--', alpha=0.2)
ax.grid(True, which='both', axis='x', ls='--', alpha=0.2)
fig.savefig(os.path.join(SAVE_DIR, "fig_energy_spectra.png"), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## Patch lattice artifact and the role of sigma_w

In [ ]:
def directional_power(field):
    P = np.abs(np.fft.fft2(field)) ** 2
    return P.mean(axis=0), P.mean(axis=1)


def peak_ratio(prof, k, halfwidth=1, floor=(4, 200)):
    band = prof[max(k - halfwidth, 1): k + halfwidth + 1]
    med = np.median(prof[floor[0]:floor[1]])
    return float(band.max() / med) if med > 0 else np.nan


def residual(pred, truth):
    return pred - truth


def lattice_k(stride, n=512):
    return n // stride


def test_remedy(stride=32, t0=16, sigma_ws=(0.35, 0.5, 0.75, 1.5, 10000.0),
                 snaps=(129, 145, 161), seed=20):
    kL = lattice_k(stride)
    rows = []
    for sw in tqdm(sigma_ws, desc="sigma_w"):
        for ti in snaps:
            prep = lambda a: (torch.from_numpy(a[:, :, ti, 0]).unsqueeze(0).float().cuda()) / STD
            cond, truth = prep(cond_data), prep(truth_data)
            f = run_sdedit(model_cond, cond, stride, t0, seed=seed, sigma_w=sw, **BASE)
            R = residual(f.squeeze().detach().cpu().numpy(), truth.squeeze().cpu().numpy())
            px, py = directional_power(R)
            band_total = float(px[8:64].sum())
            rows.append({"sigma_w": sw, "ti": ti,
                         "peak": 0.5 * (peak_ratio(px, kL) + peak_ratio(py, kL)),
                         "band_power": band_total})
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    g = df.groupby("sigma_w")[["peak", "band_power"]].mean()
    ref = g.loc[max(sigma_ws)]
    print(f"stride {stride}, t0 {t0}, lattice k = {kL}")
    print(f"{'sigma_w':>9} {'peak':>8} {'peak rel':>9} {'band rel':>9} {'selective?':>11}")
    for sw, r in g.iterrows():
        pr, br = r.peak / ref.peak, r.band_power / ref.band_power
        sel = "yes" if pr < br * 0.8 else ("no" if pr > br else "partial")
        print(f"{sw:>9g} {r.peak:>8.2f} {pr:>9.3f} {br:>9.3f} {sel:>11}")
    df.to_csv(os.path.join(SAVE_DIR, "lattice_test_remedy.csv"), index=False)
    return df


def lattice_figure(stride=BEST_STRIDE, t0=BEST_T0, ti=159, seed=20,
                    compare_strides=(16, 32, 64)):
    kL = lattice_k(stride)
    prep = lambda a: (torch.from_numpy(a[:, :, ti, 0]).unsqueeze(0).float().cuda()) / STD
    cond, truth = prep(cond_data), prep(truth_data)
    D = truth.squeeze().cpu().numpy()

    fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))
    for a, sw, lab in [(ax[0], 10000.0, "flat"), (ax[1], SIGMA_W, f"sigma_w={SIGMA_W}")]:
        f = run_sdedit(model_cond, cond, stride, t0, seed=seed, sigma_w=sw, **BASE)
        R = residual(f.squeeze().detach().cpu().numpy(), D)
        px, _ = directional_power(R)
        a.semilogy(np.arange(1, 200), px[1:200], color="#333", lw=1.1)
        for h in range(1, 200 // kL + 1):
            a.axvline(h * kL, color="#B22222", ls=":", lw=1.2,
                       label="predicted lattice k=N/s" if h == 1 else None)
        a.set_title(f"residual power, stride {stride}, t0={t0}, {lab}", fontsize=10)
        a.set_xlabel("k_x", fontsize=12)
        a.set_ylabel("power", fontsize=12)
        a.legend(fontsize=8)
        a.grid(True, ls="--", alpha=0.25)

    for s in compare_strides:
        f = run_sdedit(model_cond, cond, s, t0, seed=seed, sigma_w=10000.0, **BASE)
        R = residual(f.squeeze().detach().cpu().numpy(), D)
        px, _ = directional_power(R)
        ax[2].semilogy(np.arange(1, 200), px[1:200] / np.median(px[4:200]),
                       lw=1.2, label=f"stride {s}  (k={lattice_k(s)})")
        ax[2].axvline(lattice_k(s), ls=":", lw=1.0, alpha=0.6)
    ax[2].set_title("the peak moves with stride, as predicted", fontsize=10)
    ax[2].set_xlabel("k_x", fontsize=12)
    ax[2].set_ylabel("power / median", fontsize=12)
    ax[2].legend(fontsize=8)
    ax[2].grid(True, ls="--", alpha=0.25)

    for a in ax:
        for sp in a.spines.values():
            sp.set_edgecolor("black")
            sp.set_linewidth(1.5)
    fig.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, "fig_lattice_artifact.png"), dpi=300, bbox_inches="tight", facecolor="white")
    return fig


remedy_df = test_remedy()
fig = lattice_figure()
plt.show()

## Selecting stride and SDEdit start time

In [ ]:
def sweep_2d(model, cond, truth, strides=PAPER_STRIDES, sdedit_times=PAPER_SDEDIT,
             seed=20, spec_der=None, ref=None, kmax=128, sigma_w=10000.0, **kw):
    grid = {}
    total = len(strides) * len(sdedit_times)
    with tqdm(total=total, desc="2D sweep") as bar:
        for s in strides:
            for st in sdedit_times:
                f = run_sdedit(model, cond, s, st, seed=seed, sigma_w=sigma_w, **kw)
                rec = {"stride": s, "sdedit_time": st, "n_patches": n_patches(s, cond.shape[-1]),
                       "err": field_errors(f, truth)["notebook_norm"]}
                if spec_der is not None and ref is not None:
                    rec["spec"] = spectral_errors(f, spec_der, ref, kmax=kmax)["spec_log_norm"]
                grid[(s, st)] = rec
                torch.cuda.empty_cache()
                bar.update(1)
    return grid


def _matrix(grid, key, strides, sdedit_times):
    return np.array([[grid[(s, t)][key] for t in sdedit_times] for s in strides])


def plot_2d(grid, strides=PAPER_STRIDES, sdedit_times=PAPER_SDEDIT,
            fdns_err=None, fdns_spec=None, savepath=None):
    E = _matrix(grid, "err", strides, sdedit_times)
    has_spec = "spec" in next(iter(grid.values()))
    S = _matrix(grid, "spec", strides, sdedit_times) if has_spec else None

    ncol = 2 if has_spec else 1
    fig, axes = plt.subplots(1, ncol, figsize=(7.2 * ncol, 5.4), squeeze=False)
    axes = axes[0]

    panels = [(E, "Error Norm", fdns_err)]
    if has_spec:
        panels.append((S, "Spectral Error", fdns_spec))

    for idx, (ax, (M, label, base)) in enumerate(zip(axes, panels)):
        vmax = 8 if idx == 0 else None
        im = ax.imshow(M, origin="lower", aspect="auto", cmap="magma_r", vmax=vmax,
                        extent=[-0.5, len(sdedit_times) - 0.5, -0.5, len(strides) - 0.5])
        ax.set_xticks(range(len(sdedit_times)))
        ax.set_xticklabels(sdedit_times)
        ax.set_yticks(range(len(strides)))
        ax.set_yticklabels(strides)
        ax.set_xlabel("SDEdit start time", fontsize=12)
        ax.set_ylabel("Stride", fontsize=12)
        ax.set_title(label, fontsize=12, fontfamily="serif", fontstyle="italic")
        cb = fig.colorbar(im, ax=ax, fraction=0.046)
        if base is not None:
            ax.contour(np.arange(len(sdedit_times)), np.arange(len(strides)), M,
                       levels=[base], colors="#1F4E79", linewidths=2.0, linestyles="--")
            cb.ax.axhline(base, color="#1F4E79", lw=2)
        for i in range(len(strides)):
            for j in range(len(sdedit_times)):
                v = M[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                        color="white" if v > np.nanpercentile(M, 60) else "black")

    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches="tight", facecolor="white")
    return fig, axes


def best_per_stride(grid, strides=PAPER_STRIDES, sdedit_times=PAPER_SDEDIT, key="err"):
    print(f"{'stride':>7} {'best t0':>8} {'best':>8} {'peak t0':>8} {'peak':>8} {'ratio':>7}")
    rows = []
    for s in strides:
        vals = [(t, grid[(s, t)][key]) for t in sdedit_times if t > 0]
        bt, bv = min(vals, key=lambda p: p[1])
        pt, pv = max(vals, key=lambda p: p[1])
        rows.append({"stride": s, "best_t0": bt, "best": bv, "peak_t0": pt, "peak": pv, "ratio": pv / bv})
        print(f"{s:>7} {bt:>8} {bv:>8.3f} {pt:>8} {pv:>8.3f} {pv/bv:>7.2f}x")
    return rows


ref = reference_spectra(truth_data_T, spec_der_DNS)
grid = sweep_2d(model_cond, cond_data_T, truth_data_T, spec_der=spec_der_DNS, ref=ref, **BASE)

fdns_e = field_errors(cond_data_T, truth_data_T)["notebook_norm"]
fdns_s = spectral_errors(cond_data_T, spec_der_DNS, ref, kmax=128)["spec_log_norm"]
fig, _ = plot_2d(grid, fdns_err=fdns_e, fdns_spec=fdns_s,
                  savepath=os.path.join(SAVE_DIR, "fig_sweep_2d.png"))
plt.show()

best_per_stride(grid)

## Timing benchmarks

In [ ]:
def timed_run(fn, n_repeat=3, warmup=1):
    out = None
    for _ in range(warmup):
        out = fn()
    torch.cuda.synchronize()

    times = []
    for _ in range(n_repeat):
        torch.cuda.synchronize()
        t0 = timeit.default_timer()
        out = fn()
        torch.cuda.synchronize()
        times.append(timeit.default_timer() - t0)

    times = np.asarray(times, dtype=float)
    return out, {
        "t_min": float(times.min()),
        "t_mean": float(times.mean()),
        "t_std": float(times.std(ddof=1)) if len(times) > 1 else 0.0,
        "t_all": times.tolist(),
        "n_repeat": int(n_repeat),
    }


def _record(stride, sdedit_time, domain, timing, out, truth, spec_der, ref, kmax, num_corrections):
    rec = {"stride": stride, "sdedit_time": sdedit_time, "n_patches": n_patches(stride, domain)}
    rec["n_patch_evals"] = rec["n_patches"] * sdedit_time * (1 + num_corrections)
    rec.update(timing)
    rec.update(field_errors(out, truth))
    if spec_der is not None and ref is not None:
        rec.update(spectral_errors(out, spec_der, ref, kmax=kmax))
    return rec


def sweep_stride(model, cond, truth, strides=PAPER_STRIDES, sdedit_time=BEST_T0,
                  n_repeat=3, seed=20, spec_der=None, ref=None, kmax=None, num_corrections=1, **kw):
    records = []
    for stride in tqdm(sorted(strides), desc=f"stride sweep (t={sdedit_time})"):
        try:
            out, timing = timed_run(
                lambda: run_sdedit(model, cond, stride, sdedit_time, seed=seed, **kw), n_repeat=n_repeat)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"OOM at stride={stride}, lower chunk and rerun")
            continue
        records.append(_record(stride, sdedit_time, cond.shape[-1], timing, out, truth,
                                spec_der, ref, kmax, num_corrections))
        torch.cuda.empty_cache()
    return records


def sweep_sdedit(model, cond, truth, sdedit_times=PAPER_SDEDIT, stride=BEST_STRIDE,
                  n_repeat=3, seed=20, spec_der=None, ref=None, kmax=None, num_corrections=1, **kw):
    records = []
    for st in tqdm(sorted(sdedit_times), desc=f"sdedit sweep (stride={stride})"):
        try:
            out, timing = timed_run(
                lambda: run_sdedit(model, cond, stride, st, seed=seed, **kw), n_repeat=n_repeat)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"OOM at sdedit_time={st}, lower chunk and rerun")
            continue
        records.append(_record(stride, st, cond.shape[-1], timing, out, truth,
                                spec_der, ref, kmax, num_corrections))
        torch.cuda.empty_cache()
    return records


def tradeoff_summary(records, metric="notebook_norm", tol=0.05):
    recs = sorted(records, key=lambda r: r["stride"])
    e0, t0, s0 = recs[0][metric], recs[0]["t_min"], recs[0]["stride"]

    best = recs[0]
    for r in recs:
        if (r[metric] - e0) / e0 <= tol:
            best = r

    print(f"reference (finest stride {s0}): error = {e0:.3f}, time = {t0:.2f} s, patches = {recs[0]['n_patches']}")
    print(f"largest stride within {tol:.0%} of that error: {best['stride']}")
    print(f"  error  = {best[metric]:.3f}  ({(best[metric]-e0)/e0:+.1%})")
    print(f"  time   = {best['t_min']:.2f} s  ({t0/best['t_min']:.1f}x faster)")
    print(f"  patches= {best['n_patches']}  ({recs[0]['n_patches']/best['n_patches']:.1f}x fewer)")

    print(f"\n{'stride':>7} {'patches':>8} {'time[s]':>9} {'err':>8} {'err/ref':>8} {'speedup':>8}")
    for r in recs:
        print(f"{r['stride']:>7} {r['n_patches']:>8} {r['t_min']:>9.2f} "
              f"{r[metric]:>8.3f} {r[metric]/e0:>8.2f} {t0/r['t_min']:>8.1f}")
    return best


def metrics_table(records, x="stride", csv_path=None):
    cols = ["n_patches", "n_patch_evals", "t_min", "t_std", "notebook_norm", "spec_log_norm"]
    recs = sorted(records, key=lambda r: r[x])
    cols = [c for c in cols if all(c in r for r in recs)]
    df = pd.DataFrame([{c: r[c] for c in cols} for r in recs], index=[r[x] for r in recs])
    df.index.name = x
    if csv_path:
        df.to_csv(csv_path)
    return df


ERR_COLOR = "#B22222"
TIME_COLOR = "#1F4E79"


def plot_error_and_time(records, metric="notebook_norm", flat_region=(48, 64),
                         time_log=True, savepath=None):
    recs = sorted(records, key=lambda r: r["stride"])
    s = np.array([r["stride"] for r in recs], dtype=float)
    e = np.array([r[metric] for r in recs], dtype=float)
    t = np.array([r["t_min"] for r in recs], dtype=float)
    terr = np.array([r["t_std"] for r in recs], dtype=float)

    fig, ax1 = plt.subplots(figsize=(7.5, 4.6))
    if flat_region is not None:
        ax1.axvspan(flat_region[0], flat_region[1], color="grey", alpha=0.12, zorder=0)

    l1 = ax1.plot(s, e, "o-", color="black", markerfacecolor=ERR_COLOR, markeredgecolor=ERR_COLOR,
                  markersize=6, lw=1.4, label="Error norm", zorder=3)

    ax2 = ax1.twinx()
    l2 = ax2.errorbar(s, t, yerr=terr, fmt="s--", color=TIME_COLOR, markerfacecolor=TIME_COLOR,
                       markersize=5, lw=1.4, capsize=3, label="Wall time", zorder=3)
    if time_log:
        ax2.set_yscale("log")

    ax1.set_xlabel("Stride", fontsize=14)
    ax1.set_ylabel("Error Norm", fontsize=14, color=ERR_COLOR)
    ax2.set_ylabel("Wall time per field (s)", fontsize=14, color=TIME_COLOR)
    ax1.tick_params(axis="y", colors=ERR_COLOR, labelsize=11)
    ax2.tick_params(axis="y", colors=TIME_COLOR, labelsize=11)
    ax1.tick_params(axis="x", labelsize=11)
    ax1.xaxis.set_major_locator(FixedLocator(s))
    ax1.set_xticklabels([f"{int(v)}" for v in s])
    for sp in ax1.spines.values():
        sp.set_edgecolor("black")
        sp.set_linewidth(1.6)
    ax1.grid(True, ls="--", alpha=0.25, zorder=0)
    ax1.set_axisbelow(True)
    ax1.legend([l1[0], l2], ["Error norm", "Wall time"], fontsize=11, loc="upper left", framealpha=0.95)

    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches="tight", facecolor="white")
    return fig, (ax1, ax2)


def plot_error_vs_time(records, metric="notebook_norm", baseline=None, savepath=None):
    recs = sorted(records, key=lambda r: r["t_min"])
    t = [r["t_min"] for r in recs]
    e = [r[metric] for r in recs]

    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    ax.plot(t, e, "o-", color="black", markerfacecolor=ERR_COLOR, markeredgecolor=ERR_COLOR,
            markersize=6, lw=1.4)
    for ti, ei, r in zip(t, e, recs):
        ax.annotate(f"{r['stride']}", (ti, ei), fontsize=9, xytext=(5, 5), textcoords="offset points")
    if baseline is not None:
        ax.axhline(baseline, color="grey", ls="-.", lw=1.5, label="FDNS")
        ax.legend(fontsize=11)
    ax.set_xscale("log")
    ax.set_xlabel("Wall time per field (s)", fontsize=14)
    ax.set_ylabel("Error Norm", fontsize=14)
    ax.grid(True, which="both", ls="--", alpha=0.25)
    ax.set_axisbelow(True)
    for sp in ax.spines.values():
        sp.set_edgecolor("black")
        sp.set_linewidth(1.6)
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches="tight", facecolor="white")
    return fig, ax


ref = reference_spectra(truth_data_T, spec_der_DNS)
fdns_norm = field_errors(cond_data_T, truth_data_T)["notebook_norm"]
print(f"FDNS baseline error norm: {fdns_norm:.3f}")

recs_stride = sweep_stride(model_cond, cond_data_T, truth_data_T, strides=PAPER_STRIDES,
                            sdedit_time=BEST_T0, n_repeat=3, spec_der=spec_der_DNS, ref=ref,
                            sigma_w=SIGMA_W, **BASE)

recs_sdedit = sweep_sdedit(model_cond, cond_data_T, truth_data_T, sdedit_times=PAPER_SDEDIT,
                            stride=BEST_STRIDE, n_repeat=3, spec_der=spec_der_DNS, ref=ref,
                            sigma_w=SIGMA_W, **BASE)

np.save(os.path.join(SAVE_DIR, f"records_stride_t{BEST_T0}.npy"), np.array(recs_stride, dtype=object))
np.save(os.path.join(SAVE_DIR, f"records_sdedit_s{BEST_STRIDE}.npy"), np.array(recs_sdedit, dtype=object))

tradeoff_summary(recs_stride)

df_s = metrics_table(recs_stride, "stride", csv_path=os.path.join(SAVE_DIR, "table_vs_stride.csv"))
df_t = metrics_table(recs_sdedit, "sdedit_time", csv_path=os.path.join(SAVE_DIR, "table_vs_sdedit.csv"))
print(df_s.round(4).to_string())
print(df_t.round(4).to_string())

fig, _ = plot_error_and_time(recs_stride, savepath=os.path.join(SAVE_DIR, "fig_error_and_time.png"))
plt.show()

fig, _ = plot_error_vs_time(recs_stride, baseline=fdns_norm,
                             savepath=os.path.join(SAVE_DIR, "fig_error_vs_time.png"))
plt.show()

## Timing: single-configuration wall time

In [ ]:
_, timing = timed_run(
    lambda: run_sdedit(model_cond, cond_data_T, stride=BEST_STRIDE, sdedit_time=BEST_T0,
                        seed=20, sigma_w=SIGMA_W, **BASE),
    n_repeat=3,
)
print(f"stride={BEST_STRIDE}, t0={BEST_T0}, sigma_w={SIGMA_W}: "
      f"{timing['t_min']:.3f} s per field (mean {timing['t_mean']:.3f} +/- {timing['t_std']:.3f})")

## CRPS ensemble evaluation

In [ ]:
def crps_ensemble(members, truth, fair=True):
    x = np.asarray(members, dtype=np.float64)
    y = np.asarray(truth, dtype=np.float64)
    m = x.shape[0]
    if m < 2:
        raise ValueError("CRPS needs at least 2 members")
    term1 = np.abs(x - y).mean(axis=0)
    xs = np.sort(x, axis=0)
    k = np.arange(m).reshape(-1, 1, 1)
    pair = 2.0 * ((2 * k - m + 1) * xs).sum(axis=0)
    denom = 2.0 * m * (m - 1) if fair else 2.0 * m * m
    return term1 - pair / denom


def crps_components(members, truth, fair=True):
    y = _as_field(truth)
    m = members.shape[0]
    acc = np.abs(members - y).mean(axis=0)
    xs = np.sort(members.astype(np.float64), axis=0)
    k = np.arange(m).reshape(-1, 1, 1)
    pair = 2.0 * ((2 * k - m + 1) * xs).sum(axis=0)
    spread = pair / (2.0 * m * (m - 1) if fair else 2.0 * m * m)
    return acc, spread


def generate_ensemble(model, cond, stride=BEST_STRIDE, sdedit_time=BEST_T0, seeds=range(20, 36), **kw):
    members = []
    for sd in tqdm(list(seeds), desc="ensemble"):
        members.append(_as_field(run_sdedit(model, cond, stride, sdedit_time, seed=sd, **kw)))
        torch.cuda.empty_cache()
    return np.stack(members)


def ensemble_report(members, truth, fdns=None, fair=True):
    y = _as_field(truth)
    crps_map = crps_ensemble(members, y, fair=fair)
    mean_field = members.mean(axis=0)
    spread = members.std(axis=0, ddof=1)
    rmse_mean = float(np.sqrt(np.mean((mean_field - y) ** 2)))
    out = {
        "m": int(members.shape[0]),
        "crps": float(crps_map.mean()),
        "crps_estimator": "fair" if fair else "biased",
        "rmse_ens_mean": rmse_mean,
        "mean_spread": float(spread.mean()),
        "spread_skill_ratio": float(spread.mean() / rmse_mean) if rmse_mean else np.nan,
        "mean_member_rmse": float(np.sqrt(((members - y) ** 2).mean(axis=(1, 2))).mean()),
    }
    if fdns is not None:
        out["crps_fdns_deterministic"] = float(np.abs(_as_field(fdns) - y).mean())
    return out, crps_map, mean_field, spread


def crps_vs_size(members, truth, sizes=None, n_draws=8, seed=0):
    y = _as_field(truth)
    m = members.shape[0]
    sizes = sizes or [k for k in range(2, m + 1)]
    rng = np.random.default_rng(seed)
    out = []
    for k in sizes:
        vals = []
        for _ in range(1 if k == m else n_draws):
            idx = rng.choice(m, size=k, replace=False)
            vals.append(crps_ensemble(members[idx], y, fair=True).mean())
        out.append((k, float(np.mean(vals)), float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0))
    return out


def rank_histogram(members, truth, subsample=4):
    y = _as_field(truth)[::subsample, ::subsample]
    x = members[:, ::subsample, ::subsample]
    ranks = (x < y[None]).sum(axis=0)
    counts = np.bincount(ranks.ravel(), minlength=members.shape[0] + 1)
    return counts / counts.sum()

## CRPS plots

In [ ]:
def plot_crps_panels(members, truth, fdns=None, savepath=None):
    y = _as_field(truth)
    acc, spr = crps_components(members, y)
    crps_map = acc - spr

    fig, axes = plt.subplots(2, 2, figsize=(12.5, 9))

    ax = axes[0, 0]
    pts = crps_vs_size(members, y)
    ks = [p[0] for p in pts]
    ax.errorbar(ks, [p[1] for p in pts], yerr=[p[2] for p in pts], fmt="o-", color="black",
                markerfacecolor=ERR_COLOR, markeredgecolor=ERR_COLOR, ms=6, lw=1.5, capsize=3,
                label="ensemble CRPS")
    if fdns is not None:
        base = float(np.abs(_as_field(fdns) - y).mean())
        ax.axhline(base, color="grey", ls="-.", lw=1.6, label=f"FDNS (deterministic) = {base:.3f}")
    ax.set_xlabel("ensemble size", fontsize=12)
    ax.set_ylabel("CRPS", fontsize=12)
    ax.legend(fontsize=9)
    ax.set_title("does the ensemble size saturate?", fontsize=12)

    ax = axes[0, 1]
    freq = rank_histogram(members, y)
    ax.bar(np.arange(len(freq)), freq, color=TIME_COLOR, alpha=0.8, edgecolor="black", lw=0.6)
    ax.axhline(1.0 / len(freq), color=ERR_COLOR, ls="--", lw=1.8, label="flat = calibrated")
    ax.set_xlabel("rank of truth among members", fontsize=12)
    ax.set_ylabel("frequency", fontsize=12)
    ax.legend(fontsize=9)
    edge_mass = freq[0] + freq[-1]
    ax.set_title(f"rank histogram, {edge_mass:.0%} of truth outside the ensemble", fontsize=11)

    ax = axes[1, 0]
    labels = ["accuracy\nterm", "spread\nterm", "CRPS"]
    vals = [acc.mean(), spr.mean(), crps_map.mean()]
    colors = [ERR_COLOR, TIME_COLOR, "#2E7D32"]
    bars = ax.bar(labels, vals, color=colors, alpha=0.85, edgecolor="black", lw=0.8)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom", fontsize=10)
    if fdns is not None:
        ax.axhline(float(np.abs(_as_field(fdns) - y).mean()), color="grey", ls="-.", lw=1.6, label="FDNS")
        ax.legend(fontsize=9)
    ax.set_ylabel("domain mean", fontsize=12)
    ax.set_title("CRPS = accuracy - spread", fontsize=12)

    ax = axes[1, 1]
    im = ax.imshow(crps_map, origin="lower", cmap="viridis")
    fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"CRPS field (mean {crps_map.mean():.4f})", fontsize=12)

    for ax in axes.ravel()[:3]:
        ax.grid(True, ls="--", alpha=0.25)
        ax.set_axisbelow(True)
    for ax in axes.ravel():
        for sp in ax.spines.values():
            sp.set_edgecolor("black")
            sp.set_linewidth(1.5)

    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches="tight", facecolor="white")
    return fig, axes

## CRPS: generate the ensemble and report

In [ ]:
N_MEMBERS = 16

members = generate_ensemble(model_cond, cond_data_T, stride=BEST_STRIDE, sdedit_time=BEST_T0,
                             seeds=range(20, 20 + N_MEMBERS), sigma_w=SIGMA_W, **BASE)
np.save(os.path.join(SAVE_DIR, "ensemble_members.npy"), members)

stats, crps_map, ens_mean, spread = ensemble_report(members, truth_data_T, fdns=cond_data_T, fair=True)
np.save(os.path.join(SAVE_DIR, "crps_map.npy"), crps_map)

print(f"members            {stats['m']}")
print(f"CRPS               {stats['crps']:.4f}")
print(f"CRPS, FDNS det.    {stats['crps_fdns_deterministic']:.4f}")
print(f"RMSE of ens. mean  {stats['rmse_ens_mean']:.4f}")
print(f"mean spread        {stats['mean_spread']:.4f}")
print(f"spread-skill ratio {stats['spread_skill_ratio']:.4f}")

fig, _ = plot_crps_panels(members, truth_data_T, fdns=cond_data_T,
                           savepath=os.path.join(SAVE_DIR, "fig_crps_panels.png"))
plt.show()

## Generalization check at the ideal condition

In [ ]:
TIME_INDICES = [130, 145, 160, 175, 190]

results = []
for ti in TIME_INDICES:
    prep = lambda a: (torch.from_numpy(a[:, :, ti, 0]).unsqueeze(0).float().cuda()) / STD
    cond, truth = prep(cond_data), prep(truth_data)

    field = run_sdedit(model_cond, cond, BEST_STRIDE, BEST_T0, seed=20, sigma_w=SIGMA_W, **BASE)

    err = field_errors(field, truth)
    ref_ti = reference_spectra(truth, spec_der_DNS)
    spec = spectral_errors(field, spec_der_DNS, ref_ti, kmax=128)
    fdns_err_ti = field_errors(cond, truth)

    results.append({
        "time_index": ti,
        "notebook_norm": err["notebook_norm"],
        "rmse": err["rmse"],
        "spec_log_norm": spec["spec_log_norm"],
        "fdns_norm": fdns_err_ti["notebook_norm"],
    })
    torch.cuda.empty_cache()

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nerror norm     : mean={df['notebook_norm'].mean():.4f}  std={df['notebook_norm'].std():.4f}")
print(f"spectral error : mean={df['spec_log_norm'].mean():.4f}  std={df['spec_log_norm'].std():.4f}")

df.to_csv(os.path.join(SAVE_DIR, "ideal_condition_generalization.csv"), index=False)